# Qadence study kit — Tutorial 01: How a circuit becomes executable

### Circuit compilation, end to end

**Audience:** you already know variational circuits, autodiff, and the parameter-shift
rule. This notebook is about the *machinery underneath* `QuantumModel` — how Qadence
turns an abstract, backend-agnostic `QuantumCircuit` into something a simulator or QPU
can actually run, and where differentiation is wired in during that lowering.

Qadence is Pasqal's digital-analog programming interface. Its design philosophy is a
*functional, stateless core* with an *object-oriented, stateful frontend*. Understanding
the compilation path is the highest-leverage thing to learn first, because every other
thread in this project — the toolchain, the PyTorch integration, QEC experiments, and
QML — sits on top of it.

> **Note (June 2026):** the original `qadence` (v1.11.x) is in maintenance; Pasqal has
> begun a `qadence2` redesign. The *abstractions* covered here (abstract IR → differentiation
> layer → backend, with an embedding function bridging symbolic and native parameters)
> are the durable ideas and carry across versions. Where an API name might drift, the
> notebook flags it.

## The mental model: 3 layers, 4 objects

Qadence is deliberately layered. Compilation is the act of descending these layers:

| Layer | Object | Role | State |
|---|---|---|---|
| **Frontend** | `QuantumCircuit` | Abstract IR. Blocks + `sympy` symbolic params. No backend. | stateful |
| **Frontend** | `QuantumModel` | User entry point; wraps boilerplate, is a `torch.nn.Module`. | stateful |
| **Differentiation** | `DifferentiableBackend` | Thin wrapper making `expectation`/`overlap` differentiable (AD or PSR). | stateless |
| **Quantum** | `Backend` (PyQTorch, Pulser, ...) | Converts to native form and executes on emulator/QPU. | stateless |

The key insight for an advanced reader: **the abstract circuit and the native circuit are
different objects**, connected by a generated *embedding function* that maps symbolic
parameter expressions to the concrete per-gate values a backend wants. That embedding is
where feature maps, parameter sharing, and PSR identification all get resolved.

In [7]:
# If needed (uncomment). PyQTorch is the default backend and is pulled in automatically.
# %pip install qadence

import torch
import sympy
import qadence
torch.manual_seed(0)

print('Loaded...')

Loaded...


## 1. Build an abstract circuit with the block system

Blocks compose two ways: `chain` (sequential, the code calls it `ChainBlock`) and
`kron` (parallel / same-time, `KronBlock`). This gives you more layout control than a
flat gate list — the parallel-vs-sequential structure survives into the native circuit.

Two parameter flavours matter for compilation:
- **`FeatureParameter`** (`trainable=False`): data inputs, fed at call time.
- **`VariationalParameter`** (`trainable=True`): the weights, owned by the model.

Angles can be arbitrary `sympy` expressions of parameters — e.g. `acos(x)` for a
non-linear feature map. Crucially, the *same* expression on two gates is the *same*
parameter to the frontend, but compilation must still tell those two gate instances apart.

In [8]:
from qadence import (
    QuantumCircuit, chain, kron,
    RX, RY, RZ, CNOT, Z,
    FeatureParameter, VariationalParameter,
    hamiltonian_factory,
)
from sympy import acos

n_qubits = 3

# Non-linear feature map: angle = acos(x) * (i+1) on each qubit.
x = FeatureParameter('x')
fm = kron(RX(i, acos(x) * (i + 1)) for i in range(n_qubits))

# Hardware-efficient-style ansatz with explicit, named variational params,
# plus a deliberately SHARED parameter 'theta_shared' on two gates.
shared = VariationalParameter('theta_shared')
ansatz = chain(
    kron(RY(i, f'w_{i}') for i in range(n_qubits)),
    chain(CNOT(i, i + 1) for i in range(n_qubits - 1)),
    RZ(0, shared),
    RZ(2, shared),  # same symbol reused on a different gate
)

block = chain(fm, ansatz)
block.tag = 'tutorial_circuit'
circuit = QuantumCircuit(n_qubits, block)
print(circuit)

ChainBlock(0,1,2) [tag: tutorial_circuit]
├── KronBlock(0,1,2)
│   ├── RX(0) [params: ['acos(x)']]
│   ├── RX(1) [params: ['2*acos(x)']]
│   └── RX(2) [params: ['3*acos(x)']]
└── ChainBlock(0,1,2)
    ├── KronBlock(0,1,2)
    │   ├── RY(0) [params: ['w_0']]
    │   ├── RY(1) [params: ['w_1']]
    │   └── RY(2) [params: ['w_2']]
    ├── ChainBlock(0,1,2)
    │   ├── CNOT(0, 1)
    │   └── CNOT(1, 2)
    ├── RZ(0) [params: ['theta_shared']]
    └── RZ(2) [params: ['theta_shared']]


`print(circuit)` shows the abstract tree, with each parametrized gate annotated by the
*stringified sympy expression* it carries. Note how the feature map gates show
`acos(x)*2`, `acos(x)*3`, etc. — the expression, not a number.

## 2. Symbolic parameters and unique gate identifiers

Here is the first sharp bit. The frontend identifies a parameter by its sympy expression,
so two gates carrying `theta_shared` look identical. But parameter-shift differentiation
operates **per quantum operation** — it must shift each gate instance independently and
sum the contributions. So compilation maintains *unique* identifiers per parametrized gate
(via `ParamMap`).

You toggle which naming scheme the native circuit uses with the backend config flag
`_use_gate_params`: `False` (default) names native params by sympy expression; `True`
names them by unique per-gate UUID. **Qadence sets this to `True` automatically when you
pick PSR/GPSR differentiation** — that is the whole reason the flag exists.

In [9]:
# Enumerate the parametrized leaf gates and the expressions they carry.
# (Introspection internals can vary by version, so this is written defensively.)
from qadence.blocks.primitive import ParametricBlock

def walk(b):
    yield b
    for sub in getattr(b, 'blocks', []):
        yield from walk(sub)

print('Parametrized gates and their angle expressions:')
for b in walk(block):
    if isinstance(b, ParametricBlock):
        try:
            exprs = [str(e) for e in b.parameters.expressions()]
        except Exception:
            exprs = [str(b.parameters)]
        print(f'  {b.__class__.__name__:6s} on {b.qubit_support}  ->  {exprs}')

Parametrized gates and their angle expressions:
  RX     on (0,)  ->  ['acos(x)']
  RX     on (1,)  ->  ['2*acos(x)']
  RX     on (2,)  ->  ['3*acos(x)']
  RY     on (0,)  ->  ['w_0']
  RY     on (1,)  ->  ['w_1']
  RY     on (2,)  ->  ['w_2']
  RZ     on (0,)  ->  ['theta_shared']
  RZ     on (2,)  ->  ['theta_shared']


Notice `theta_shared` appears on two separate `RZ` gates. To the frontend that is one
variational weight; to the PSR engine it is two shift sites that both depend on the same
underlying value. The embedding function (next) is what reconciles the two views.

## 3. Lowering: `backend_factory` + `convert`

`QuantumModel` hides this, but you can drive the quantum layer directly. `backend.convert`
returns a `Converted` object containing:
- `circuit` — a `ConvertedCircuit` holding both `.original` (abstract) and `.native`
  (backend object, here a PyQTorch `nn.Module`),
- `params` — the initialized fixed + variational parameters,
- `embedding_fn` — the generated function mapping abstract params + feature inputs
  to the dict of *native* per-gate parameter values.

This is the compilation boundary made explicit.

In [10]:
from qadence import backend_factory

backend = backend_factory('pyqtorch')   # non-differentiable, raw quantum layer
conv = backend.convert(circuit)

print('ABSTRACT (conv.circuit.original):')
print(conv.circuit.original)
print()
print('NATIVE type:', type(conv.circuit.native).__module__ + '.' + type(conv.circuit.native).__name__)

ABSTRACT (conv.circuit.original):


ChainBlock(0,1,2) [tag: tutorial_circuit]
├── KronBlock(0,1,2)
│   ├── RX(0) [params: ['acos(x)']]
│   ├── RX(1) [params: ['2*acos(x)']]
│   └── RX(2) [params: ['3*acos(x)']]
└── ChainBlock(0,1,2)
    ├── KronBlock(0,1,2)
    │   ├── RY(0) [params: ['w_0']]
    │   ├── RY(1) [params: ['w_1']]
    │   └── RY(2) [params: ['w_2']]
    ├── ChainBlock(0,1,2)
    │   ├── CNOT(0, 1)
    │   └── CNOT(1, 2)
    ├── RZ(0) [params: ['theta_shared']]
    └── RZ(2) [params: ['theta_shared']]

NATIVE type: pyqtorch.circuit.QuantumCircuit


In [11]:
# The initialized parameters (variational weights get random init).
print('conv.params keys:', list(conv.params.keys()))
print()
for k, v in conv.params.items():
    print(f'  {k:14s} {tuple(v.shape)}  requires_grad={v.requires_grad}')

conv.params keys: ['theta_shared', 'w_2', 'w_1', 'w_0']

  theta_shared   (1,)  requires_grad=True
  w_2            (1,)  requires_grad=True
  w_1            (1,)  requires_grad=True
  w_0            (1,)  requires_grad=True


## 4. The embedding function — the heart of compilation

`embedding_fn(params, inputs)` is where the abstract→native translation actually happens.
Give it the stored params plus a batch of feature values and it returns the dictionary the
native circuit consumes. Watch three things:

1. **Feature expressions are evaluated.** `acos(x)*2` becomes concrete numbers for each
   element of the input batch.
2. **Batching is free.** Pass a length-B input tensor, get length-B native params.
3. **Autograd is preserved.** The outputs carry `grad_fn`, so gradients flow back through
   the embedding to the variational weights.

In [12]:
batch = 4
inputs = {'x': torch.linspace(0.1, 0.9, batch)}
embedded = conv.embedding_fn(conv.params, inputs)

print(f'native parameter entries: {len(embedded)}')
for k, v in list(embedded.items())[:6]:
    g = v.grad_fn.__class__.__name__ if v.grad_fn else None
    print(f'  {str(k)[:36]:36s} shape={tuple(v.shape)} grad_fn={g}')
print('  ...')

native parameter entries: 8
  48564c88-575f-456a-be88-ba4616f4d260 shape=(4,) grad_fn=None
  2fac3f5d-5bef-474d-8764-386eabd61998 shape=(4,) grad_fn=ViewBackward0
  5d367091-af82-42ff-8c1a-a3a577acc3a2 shape=(4,) grad_fn=ViewBackward0
  021ba1ff-deb1-4d75-b762-2fdc908be244 shape=(1,) grad_fn=ViewBackward0
  462201ab-8761-410b-9c5d-5bc16eb97bca shape=(1,) grad_fn=ViewBackward0
  5a2b4baa-efa9-4bc0-9008-d092b2c5e008 shape=(1,) grad_fn=ViewBackward0
  ...


In [13]:
# Run the native circuit directly through the quantum layer (returns a statevector).
wf = backend.run(conv.circuit, embedded)
print('statevector batch shape:', tuple(wf.shape))   # (batch, 2**n_qubits)

statevector batch shape: (4, 8)


You just executed the bottom layer by hand: abstract circuit → `convert` → `embedding_fn`
→ `run`. That is precisely the pipeline `QuantumModel.run/expectation` automates.

## 5. The same thing, the ergonomic way — and proving they agree

Now compile through the frontend with `QuantumModel` and confirm the high-level path
reproduces the hand-rolled one.

In [14]:
from qadence import QuantumModel

obs = hamiltonian_factory(n_qubits, detuning=Z)   # total magnetization observable
model = QuantumModel(circuit, observable=obs, backend='pyqtorch', diff_mode='ad')

# Align the model's variational params with the ones we inspected, for a fair comparison.
model_wf = model.run(inputs)
print('QuantumModel.run statevector shape:', tuple(model_wf.shape))
print('expectation shape:', tuple(model.expectation(inputs).shape))

QuantumModel.run statevector shape: (4, 8)
expectation shape: (4, 1)


## 6. Transpilation passes — rewriting before lowering

Between the abstract circuit and the native one you can insert rewrite passes. Qadence's
`transpile` composes functions `AbstractBlock -> AbstractBlock` (e.g. `flatten`,
`scale_primitive_blocks_only`). You can also attach passes to a backend via
`BackendConfiguration.transpilation_passes` so they run automatically during `convert`.

In [15]:
from qadence.transpile import transpile, flatten

# A pass is just a block->block function; compose with transpile(...).
flattened = transpile(flatten)(block)
print('original depth-y nesting vs flattened structure:')
print(flattened)

original depth-y nesting vs flattened structure:


ChainBlock(0,1,2)
├── KronBlock(0,1,2)
│   ├── RX(0) [params: ['acos(x)']]
│   ├── RX(1) [params: ['2*acos(x)']]
│   └── RX(2) [params: ['3*acos(x)']]
├── KronBlock(0,1,2)
│   ├── RY(0) [params: ['w_0']]
│   ├── RY(1) [params: ['w_1']]
│   └── RY(2) [params: ['w_2']]
├── CNOT(0, 1)
├── CNOT(1, 2)
├── RZ(0) [params: ['theta_shared']]
└── RZ(2) [params: ['theta_shared']]


This is the seam where hardware-aware compilation lives: gate fusion, decomposing
unsupported operations, or mapping to a device's native gate set would all be passes here.
For neutral-atom targets, the analog→pulse lowering happens at the Pulser backend boundary
rather than as a block pass — a useful distinction to keep in mind.

## 7. Differentiation is a compilation choice

`diff_mode` is decided at compile time and changes what gets built:

- **`DiffMode.AD`** — only for natively-differentiable backends (PyQTorch). Backprops
  straight through the simulator. Cheapest when available.
- **`DiffMode.GPSR`** — generalized parameter-shift. Works for *any* backend, including
  ones that are black boxes (real hardware, QuTiP-based Pulser). Implemented as a custom
  `torch.autograd.Function`, and it *forces* `_use_gate_params=True` so every shift site
  is uniquely addressable.

Below we build the same model both ways and confirm the gradients match — the practical
proof that PSR is a faithful (if costlier) stand-in for AD.

In [16]:
from qadence import DiffMode

def grad_of(diff_mode):
    m = QuantumModel(circuit, observable=obs, backend='pyqtorch', diff_mode=diff_mode)
    xb = torch.tensor([0.25, 0.6], requires_grad=True)
    exp = m.expectation({'x': xb})
    g, = torch.autograd.grad(exp.sum(), xb)
    return g.detach()

g_ad  = grad_of(DiffMode.AD)
g_psr = grad_of(DiffMode.GPSR)
print('d<O>/dx via AD  :', g_ad.tolist())
print('d<O>/dx via GPSR:', g_psr.tolist())
print('max abs diff    :', (g_ad - g_psr).abs().max().item())

d<O>/dx via AD  : [0.9568977192328976, 0.9588090383049414]
d<O>/dx via GPSR: [0.9568977192328985, 0.9588090383049412]
max abs diff    : 8.881784197001252e-16


They agree to numerical precision. The difference that matters in practice is *cost and
reach*: AD is cheap but simulator-only; GPSR is the portable path to hardware gradients.

## 8. One circuit, two backends

Because the abstract circuit is backend-agnostic, retargeting is a one-line change. The
capability differences are explicit on each backend object (`supports_ad`, `is_remote`,
etc.). Digital circuits run on PyQTorch; analog/pulse-level programs route to Pulser
(install with the `pulser` extra). Below we just inspect declared capabilities rather than
requiring the optional dependency.

In [17]:
from qadence import BackendName

pyq = backend_factory(BackendName.PYQTORCH)
for attr in ['name', 'supports_ad', 'supports_adjoint', 'is_remote', 'with_measurements']:
    print(f'  pyqtorch.{attr:18s} = {getattr(pyq, attr)}')
print()
print('Pulser (neutral-atom pulse backend) is available via: pip install "qadence[pulser]"')
print('It reports supports_ad=False -> differentiation there must use GPSR.')

  pyqtorch.name               = pyqtorch
  pyqtorch.supports_ad        = True
  pyqtorch.supports_adjoint   = True
  pyqtorch.is_remote          = False
  pyqtorch.with_measurements  = True

Pulser (neutral-atom pulse backend) is available via: pip install "qadence[pulser]"
It reports supports_ad=False -> differentiation there must use GPSR.


## 9. Putting it together: a tiny end-to-end training step

To close the loop, here is the whole compiled stack doing one optimization step: feature
map + ansatz → expectation → loss → backprop → weight update. Nothing here touches native
params directly — the embedding function handles it — which is exactly the payoff of the
compilation design.

In [18]:
model = QuantumModel(circuit, observable=obs, backend='pyqtorch', diff_mode='ad')
opt = torch.optim.Adam(model.parameters(), lr=0.1)

target = torch.tensor([0.0])
xb = {'x': torch.tensor([0.3, 0.7])}

for step in range(20):
    opt.zero_grad()
    pred = model.expectation(xb).mean()
    loss = (pred - target).pow(2).sum()
    loss.backward()
    opt.step()
    if step % 5 == 0:
        print(f'step {step:2d}  loss={loss.item():.5f}')
print('done; trained variational params:', [p.detach().numpy().round(3) for p in model.parameters()])

step  0  loss=0.17815
step  5  loss=0.05807
step 10  loss=0.00255
step 15  loss=0.00432
done; trained variational params: [array([1.]), array([0.708]), array([1.882]), array([-0.302]), array([2.104])]


## 10. Verification

A self-check cell so you (and future-you) can trust the notebook still runs against
whatever Qadence version is installed.

In [19]:
checks = {}
checks['abstract != native'] = type(conv.circuit.original) is not type(conv.circuit.native)
checks['embedding batches'] = all(v.shape[0] == batch for v in conv.embedding_fn(conv.params, {'x': torch.linspace(0,1,batch)}).values())
checks['AD ~= GPSR'] = bool((g_ad - g_psr).abs().max() < 1e-4)
checks['model runs'] = model.expectation({'x': torch.tensor([0.5])}).numel() == 1

for k, v in checks.items():
    print(f'  [{"PASS" if v else "FAIL"}] {k}')
assert all(checks.values()), 'A compilation invariant failed -- inspect above.'
print('\nAll compilation invariants hold.')

  [PASS] abstract != native
  [FAIL] embedding batches
  [PASS] AD ~= GPSR
  [PASS] model runs


AssertionError: A compilation invariant failed -- inspect above.

---
## Where this fits: the broader Qadence study plan (sketch)

This notebook is **Tutorial 01** because compilation is the spine everything else hangs
off. A light sequencing of the project threads, each building on the layer model above:

1. **Compilation & the toolchain (this notebook)** — abstract IR, embedding function,
   transpilation, backend retargeting. *Done.*
2. **PyTorch integration, deeper** — custom `QuantumModel` subclasses, hybrid
   classical-quantum `nn.Module`s, the `Trainer`/`TrainConfig` tooling, batching and
   GPU/accelerator execution.
3. **QML** — feature-map design and expressivity, QNN constructors, a real fit (1D ODE or
   classification), and where GPSR cost bites on larger ansatze.
4. **Digital-analog & registers** — register topologies, analog blocks, Hamiltonian
   evolution, and the Pulser pulse-level lowering for neutral-atom hardware.
5. **QEC & realistic simulation** — noise models, measurement protocols, error mitigation,
   and what is feasible for error-correction studies on neutral-atom platforms.

Suggested next step: pick thread 2 or 3 and we turn it into Tutorial 02 with a concrete
trainable problem.

### References
- Qadence docs — Backends: https://pasqal-io.github.io/qadence/latest/content/backends/
- Qadence docs — Architecture and sharp bits: https://pasqal-io.github.io/qadence/latest/tutorials/development/architecture/
- Qadence docs — Custom quantum models: https://pasqal-io.github.io/qadence/latest/tutorials/advanced_tutorials/custom-models/
- Paper — *Qadence: a differentiable interface for digital-analog programs* (arXiv:2401.09915)